In [ ]:
#| default_exp keyword_ranking

# Keyword Ranking
> Track and analyze keyword rankings from GSC performance data.

In [ ]:
# | export
from datetime import datetime, timedelta
from sqlmodel import select, func
import altair as alt
import polars as pl
from seo_rat.gsc_client import get_date_range
from seo_rat.models import GSCAnalytics

In [ ]:
#| export
def get_previous_period(start: str,  # Start date (YYYY-MM-DD)
                        end: str  # End date (YYYY-MM-DD)
                        ) -> tuple[str, str]:
    "Return the date range immediately preceding the given range with equal duration."
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    duration = e - s
    return ((s - duration - timedelta(days=1)).strftime("%Y-%m-%d"),
            (s - timedelta(days=1)).strftime("%Y-%m-%d"))


def _fetch_period_metrics(session, site_url:str, keyword:str,
                          start:str, end:str, country:str|None=None) -> tuple:
    "Fetch avg position, total clicks, total impressions for a keyword in a date range."
    q = select(func.avg(GSCAnalytics.position),
               func.sum(GSCAnalytics.clicks),
               func.sum(GSCAnalytics.impressions)
              ).where(GSCAnalytics.site_url == site_url,
                      GSCAnalytics.query == keyword,
                      GSCAnalytics.date.between(start, end))
    if country: q = q.where(GSCAnalytics.country == country)
    return session.exec(q).first() or (None, None, None)


def _build_keyword_ranking(session, site_url:str, keyword:str,
                           start:str, end:str, prev_start:str, prev_end:str,
                           country:str|None=None) -> dict:
    "Build a keyword ranking dict with current metrics and position change vs previous period."
    cur_pos, clicks, impressions = _fetch_period_metrics(session, site_url, keyword, start, end, country)
    prev_pos = _fetch_period_metrics(session, site_url, keyword, prev_start, prev_end, country)[0]
    return {"keyword": keyword,
            "position": round(cur_pos, 1) if cur_pos else None,
            "change": round(cur_pos - prev_pos, 2) if cur_pos and prev_pos else None,
            "clicks": clicks or 0,
            "impressions": impressions or 0}


def get_keyword_rankings(session,  # Active database session
                         site_url: str,  # GSC property URL
                         keywords: list[str],  # Keywords to look up
                         range_type: str = "last_7_days",  # Date range type
                         country: str | None = None  # ISO 3166-1 alpha-3 country code
                         ) -> list[dict]:
    "Get aggregated ranking metrics for keywords with period-over-period position change."
    start, end = get_date_range(range_type=range_type)
    prev_start, prev_end = get_previous_period(start, end)
    return [_build_keyword_ranking(session, site_url, kw, start, end, prev_start, prev_end, country)
            for kw in keywords]



In [ ]:
#| hide
#| eval: false
from seo_rat.sqlite_db import get_session
from seo_rat.models import GSCAnalytics

with get_session() as session:
    test_keyword_ranking = get_keyword_rankings(
        session,
        site_url="sc-domain:awazly.com",
        keywords=["شركة عزل اسطح بجدة"],
        range_type="last_month",
        country="sau",

    )
    test_keyword_ranking

In [ ]:
#| export
def _fetch_daily_rankings(session, site_url: str, keywords: list[str],
                          start: str, end: str, country: str | None = None) -> list[dict]:
    "Fetch raw daily position, clicks and impressions rows for multiple keywords."
    q = select(GSCAnalytics.date, GSCAnalytics.query, GSCAnalytics.position,
               GSCAnalytics.clicks, GSCAnalytics.impressions
               ).where(GSCAnalytics.site_url == site_url,
                       GSCAnalytics.query.in_(keywords),
                       GSCAnalytics.date.between(start, end)
                       ).order_by(GSCAnalytics.query, GSCAnalytics.date)
    if country: q = q.where(GSCAnalytics.country == country)
    return [{"date": r.date, "keyword": r.query, "position": r.position,
             "clicks": r.clicks, "impressions": r.impressions}
            for r in session.exec(q)]


def get_keyword_rankings_daily(session,  # Active database session
                               site_url: str,  # GSC property URL
                               keywords: list[str],  # Keywords to look up
                               range_type: str = "last_7_days",  # Date range type
                               days=None,
                               months=None,
                               country: str | None = None  # ISO 3166-1 alpha-3 country code
                               ) -> list[dict]:
    "Fetch daily ranking metrics per keyword for trend charts."
    start, end = get_date_range(range_type=range_type,days=days,months=months)
    return _fetch_daily_rankings(session, site_url, keywords, start, end, country)


In [ ]:
# | export
def plot_keyword_rankings(results: list[dict]  # Output of `get_keyword_rankings_daily`
                          ) -> alt.VConcatChart:
    "Plot keyword position and clicks over time as a two-panel chart."
    df = pl.DataFrame(results).with_columns(pl.col("date").str.to_date())
    color = alt.Color("keyword:N", scale=alt.Scale(scheme="tableau10"), title="Keyword")
    x = alt.X("date:T", title=None, axis=alt.Axis(format="%b %d", labelAngle=-30))
    tooltip = [alt.Tooltip("date:T", title="Date", format="%b %d %Y"),
               alt.Tooltip("keyword:N", title="Keyword"),
               alt.Tooltip("position:Q", title="Position", format=".1f"),
               alt.Tooltip("clicks:Q", title="Clicks"),
               alt.Tooltip("impressions:Q", title="Impressions")]
    pos_chart = alt.Chart(df).mark_line(point=True, strokeWidth=2).encode(
        x=x, color=color, tooltip=tooltip,
        y=alt.Y("position:Q", scale=alt.Scale(reverse=True), title="Position",
                axis=alt.Axis(tickMinStep=1)),
    ).properties(title="Position Over Time", width=680, height=240)
    clicks_chart = alt.Chart(df).mark_bar(opacity=0.8).encode(
        x=x, color=color, tooltip=tooltip,
        y=alt.Y("clicks:Q", title="Clicks"),
    ).properties(title="Daily Clicks", width=680, height=130)
    return (alt.vconcat(pos_chart, clicks_chart)
            .configure_axis(labelFontSize=11, titleFontSize=12, gridOpacity=0.2)
            .configure_title(fontSize=13, anchor="start", fontWeight="bold")
            .configure_view(strokeWidth=0)
            .configure_legend(orient="top", labelFontSize=11))

In [ ]:
#| hide
#| eval: false
test_daily = get_keyword_rankings_daily(
    session,
    site_url="sc-domain:awazly.com",
    keywords=["شركة عزل فوم بجدة"],
    range_type="last_month",
    country="sau",
)
plot_keyword_rankings(test_daily)